In [56]:
pip install numpy pandas pyarrow scikit-learn ipython flwr "flwr[simulation]"

Note: you may need to restart the kernel to use updated packages.


In [57]:
import pandas as pd
import numpy as np
from IPython.display import display

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [58]:
# Panggil file path

file_path = "../dataset/LDAP-testing.parquet" 

# Membaca file parquet
df = pd.read_parquet(file_path, engine='pyarrow')

# Menampilkan 5 baris pertama untuk memastikan data berhasil dimuat
print("Pratinjau Data:")
display(df.head())

# Melihat informasi umum tipe data dan jumlah kolom/baris
print("\nInformasi Dataset:")
df.info()

Pratinjau Data:


,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,17,1,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,DrDoS_LDAP
1,17,1,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,...,1472,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,DrDoS_LDAP
2,17,1,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,DrDoS_LDAP
3,17,2,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,...,-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,DrDoS_LDAP
4,17,1,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,...,14,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,DrDoS_LDAP



Informasi Dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2831 entries, 0 to 2830
Data columns (total 78 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   Protocol                  2831 non-null   int8    
 1   Flow Duration             2831 non-null   int32   
 2   Total Fwd Packets         2831 non-null   int16   
 3   Total Backward Packets    2831 non-null   int16   
 4   Fwd Packets Length Total  2831 non-null   float32 
 5   Bwd Packets Length Total  2831 non-null   float32 
 6   Fwd Packet Length Max     2831 non-null   float32 
 7   Fwd Packet Length Min     2831 non-null   float32 
 8   Fwd Packet Length Mean    2831 non-null   float32 
 9   Fwd Packet Length Std     2831 non-null   float32 
 10  Bwd Packet Length Max     2831 non-null   float32 
 11  Bwd Packet Length Min     2831 non-null   float32 
 12  Bwd Packet Length Mean    2831 non-null   float32 
 13  Bwd Packet Length Std     28

**CLEANING DATA** | DDOS DATASET

In [59]:
# --- BASIC CLEANING ---

# Mengecek apakah ada nilai kosong (NaN/Null)
print("Jumlah nilai kosong per kolom:")
print(df.isnull().sum())

# Delete Null
df = df.dropna()

# Menghapus baris duplikat jika ada
df = df.drop_duplicates()

Jumlah nilai kosong per kolom:
Protocol                    0
Flow Duration               0
Total Fwd Packets           0
Total Backward Packets      0
Fwd Packets Length Total    0
                           ..
Idle Mean                   0
Idle Std                    0
Idle Max                    0
Idle Min                    0
Label                       0
Length: 78, dtype: int64


In [60]:
# --- HAPUS KOLOM NON RELEVAN ---

# Menghapus Fitur Identitas/Bias | Daftar kolom yang umumnya ada di CIC-DDOS2019 dan perlu dihapus
cols_to_drop = ['Flow ID', 'Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Timestamp']

# Hanya drop kolom yang benar-benar ada di DataFrame
existing_cols_to_drop = [col for col in cols_to_drop if col in df.columns]
df = df.drop(columns=existing_cols_to_drop)

print(f"Kolom yang dihapus: {existing_cols_to_drop}")
print(f"Sisa jumlah kolom: {df.shape[1]}")

Kolom yang dihapus: []
Sisa jumlah kolom: 78


**DATA PREPROCESSING** | DDOS DATASET

In [61]:
# --- NORMALISASI DATA ---

# Menggunakan StandardScaler agar nilai fitur memiliki rata-rata 0 dan standar deviasi 1
scaler = StandardScaler()

# Before
print("Data sebelum dinormalisasi:")
display(df.head())

# Hanya menormalisasi data numerik (mengabaikan kolom teks/kategorikal seperti IP Address jika belum di-encode)
X_numeric = df.select_dtypes(include=['number']).columns.drop('Label', errors='ignore')
df[X_numeric] = scaler.fit_transform(df[X_numeric])

# After
print("Data setelah dinormalisasi:")
display(df.head())

Data sebelum dinormalisasi:


,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,17,1,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,DrDoS_LDAP
1,17,1,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,...,1472,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,DrDoS_LDAP
2,17,1,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,DrDoS_LDAP
3,17,2,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,...,-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,DrDoS_LDAP
4,17,1,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,...,14,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,DrDoS_LDAP


Data setelah dinormalisasi:


,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,0.713179,-0.200883,-0.185137,-0.252802,0.629462,-0.101652,0.978862,1.031885,1.027821,-0.189113,...,0.167093,-0.018804,0.0,-0.018804,-0.018804,-0.041061,0.0,-0.041061,-0.041061,DrDoS_LDAP
1,0.713179,-0.200883,-0.185137,-0.252802,0.629462,-0.101652,0.978862,1.031885,1.027821,-0.189113,...,0.167102,-0.018804,0.0,-0.018804,-0.018804,-0.041061,0.0,-0.041061,-0.041061,DrDoS_LDAP
2,0.713179,-0.200883,-0.185137,-0.252802,0.629462,-0.101652,0.978862,1.031885,1.027821,-0.189113,...,0.167093,-0.018804,0.0,-0.018804,-0.018804,-0.041061,0.0,-0.041061,-0.041061,DrDoS_LDAP
3,0.713179,-0.200881,-0.185137,-0.252802,0.629462,-0.101652,0.978862,1.031885,1.027821,-0.189113,...,0.167093,-0.018804,0.0,-0.018804,-0.018804,-0.041061,0.0,-0.041061,-0.041061,DrDoS_LDAP
4,0.713179,-0.200883,-0.185137,-0.252802,0.629462,-0.101652,0.978862,1.031885,1.027821,-0.189113,...,0.167093,-0.018804,0.0,-0.018804,-0.018804,-0.041061,0.0,-0.041061,-0.041061,DrDoS_LDAP


In [62]:
# Before
df['Label'].value_counts()

Label
DrDoS_LDAP    1440
Benign        1391
Name: count, dtype: int64

In [63]:
# Inisialisasi Label Encoder
le = LabelEncoder()

# MELAKUKAN ENCODING PADA KOLOM 'Label'
df['Label'] = le.fit_transform(df['Label'])

# After
df['Label'].value_counts()

Label
1    1440
0    1391
Name: count, dtype: int64

In [64]:
df

,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,0.713179,-0.200883,-0.185137,-0.252802,0.629462,-0.101652,0.978862,1.031885,1.027821,-0.189113,...,0.167093,-0.018804,0.0,-0.018804,-0.018804,-0.041061,0.0,-0.041061,-0.041061,1
1,0.713179,-0.200883,-0.185137,-0.252802,0.629462,-0.101652,0.978862,1.031885,1.027821,-0.189113,...,0.167102,-0.018804,0.0,-0.018804,-0.018804,-0.041061,0.0,-0.041061,-0.041061,1
2,0.713179,-0.200883,-0.185137,-0.252802,0.629462,-0.101652,0.978862,1.031885,1.027821,-0.189113,...,0.167093,-0.018804,0.0,-0.018804,-0.018804,-0.041061,0.0,-0.041061,-0.041061,1
3,0.713179,-0.200881,-0.185137,-0.252802,0.629462,-0.101652,0.978862,1.031885,1.027821,-0.189113,...,0.167093,-0.018804,0.0,-0.018804,-0.018804,-0.041061,0.0,-0.041061,-0.041061,1
4,0.713179,-0.200883,-0.185137,-0.252802,0.629462,-0.101652,0.978862,1.031885,1.027821,-0.189113,...,0.167093,-0.018804,0.0,-0.018804,-0.018804,-0.041061,0.0,-0.041061,-0.041061,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2826,-1.347226,-0.076730,0.161734,0.028575,-0.787646,-0.088035,-1.028115,-1.027960,-1.021311,0.133197,...,0.167093,-0.018804,0.0,-0.018804,-0.018804,-0.041061,0.0,-0.041061,-0.041061,0
2827,-1.347226,-0.097371,0.161734,0.028575,-0.787646,-0.088035,-1.028115,-1.027960,-1.021311,0.133197,...,0.167093,-0.018804,0.0,-0.018804,-0.018804,-0.041061,0.0,-0.041061,-0.041061,0
2828,-1.347226,-0.076896,0.161734,0.028575,-0.787646,-0.088035,-1.028115,-1.027960,-1.021311,0.133197,...,0.167093,-0.018804,0.0,-0.018804,-0.018804,-0.041061,0.0,-0.041061,-0.041061,0
2829,-1.347226,-0.076977,0.161734,0.028575,-0.787646,-0.088035,-1.028115,-1.027960,-1.021311,0.133197,...,0.167093,-0.018804,0.0,-0.018804,-0.018804,-0.041061,0.0,-0.041061,-0.041061,0


In [65]:
df.to_csv("LDAP-training.csv", index=False)

**TRAINING MANUAL SIMULATION** | DDOS DATASET (Konsep Matematika dibalik Federated Learning secara Manual)

In [ ]:
X = df.drop(columns=['Label'])
y = df['Label']

In [ ]:
print("\n--- 3. Simulasi Distribusi Data ke Node Klien ---")

# Menentukan jumlah node (klien) yang akan berpartisipasi dalam FL
NUM_CLIENTS = 3

# Mengacak indeks data agar distribusi lebih merata sebelum dipotong-potong
indices = np.random.permutation(len(X))

# Membagi indeks ke dalam beberapa array sesuai jumlah klien
client_indices = np.array_split(indices, NUM_CLIENTS)

# Membuat dictionary untuk menyimpan data masing-masing klien
client_data = {}

for i in range(NUM_CLIENTS):
    # Mengambil fitur dan label berdasarkan indeks yang sudah dibagi
    client_X = X.iloc[client_indices[i]]
    client_y = y[client_indices[i]]
    
    client_data[f"client_{i+1}"] = {
        'X': client_X,
        'y': client_y
    }
    
    # Menampilkan info jumlah data per klien
    print(f"Client {i+1} menerima {len(client_X)} sampel data.")

# Contoh cara memanggil data milik klien pertama:
# X_klien_1 = client_data["client_1"]["X"]
# y_klien_1 = client_data["client_1"]["y"]

In [ ]:
import copy
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report # Tambahan impor

print("--- Memulai Simulasi Federated Learning (FedAvg) ---")

all_classes = np.array([0, 1]) 
COMMUNICATION_ROUNDS = 5
NUM_EPOCHS_LOCAL = 3 

global_model = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=1, warm_start=True, random_state=42)

# --- INI DATA DUMMY YANG SUDAH DIPERBAIKI ---
n_features = client_data["client_1"]['X'].shape[1]
kolom_asli = client_data["client_1"]['X'].columns 

dummy_X = pd.DataFrame(np.zeros((2, n_features)), columns=kolom_asli)
dummy_y = np.array([0, 1]) 

global_model.partial_fit(dummy_X, dummy_y, classes=all_classes)
# ---------------------------------------------

for round_num in range(COMMUNICATION_ROUNDS):
    print(f"\n[+] Communication Round {round_num + 1}/{COMMUNICATION_ROUNDS}")
    
    local_weights = []
    local_intercepts = []
    
    for client_id, data in client_data.items():
        X_local = data['X']
        y_local = data['y']
        
        local_model = copy.deepcopy(global_model)
        
        for _ in range(NUM_EPOCHS_LOCAL):
            local_model.partial_fit(X_local, y_local) 
            
        local_weights.append(local_model.coefs_)
        local_intercepts.append(local_model.intercepts_)
        
        print(f"    - {client_id} selesai melatih data lokal.")
        
    print("    -> Server mengagregasi bobot (Federated Averaging)...")
    
    avg_weights = []
    for layer_idx in range(len(local_weights[0])):
        layer_weights = np.mean([client_weights[layer_idx] for client_weights in local_weights], axis=0)
        avg_weights.append(layer_weights)
        
    avg_intercepts = []
    for layer_idx in range(len(local_intercepts[0])):
        layer_intercepts = np.mean([client_intercepts[layer_idx] for client_intercepts in local_intercepts], axis=0)
        avg_intercepts.append(layer_intercepts)
        
    global_model.coefs_ = avg_weights
    global_model.intercepts_ = avg_intercepts
    
    # --- EVALUASI MODEL GLOBAL (DENGAN LAPORAN LENGKAP) ---
    print("\n    => Laporan Kinerja Model Global:")
    val_preds = global_model.predict(client_data["client_1"]['X'])
    
    # Mencetak laporan komprehensif
    # parameter zero_division=0 mencegah error pembagian jika model menebak semua ke satu kelas
    report = classification_report(client_data["client_1"]['y'], val_preds, zero_division=0)
    print(report)
    
print("\n--- Pelatihan Selesai! Model Global siap dideploy. ---")

**TRAINING FLOWER SIMULATION** | DDOS DATASET

In [ ]:
import flwr as fl
import numpy as np
import pandas as pd
import copy
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
import warnings

# Menyembunyikan warning agar output terminal tetap bersih
warnings.simplefilter("ignore")

# ==========================================
# 1. INISIALISASI MODEL DASAR
# ==========================================
base_model = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=1, warm_start=True, random_state=42)

n_features = client_data["client_1"]['X'].shape[1]
dummy_X = pd.DataFrame(np.zeros((2, n_features)), columns=client_data["client_1"]['X'].columns)
dummy_y = np.array([0, 1])
base_model.partial_fit(dummy_X, dummy_y, classes=np.array([0, 1]))

# ==========================================
# 2. FUNGSI PENERJEMAH UNTUK FLOWER
# ==========================================
def get_model_parameters(model):
    return [np.copy(w) for w in model.coefs_] + [np.copy(b) for b in model.intercepts_]

def set_model_parameters(model, parameters):
    n_layers = len(model.coefs_)
    model.coefs_ = parameters[:n_layers]
    model.intercepts_ = parameters[n_layers:]
    return model

# ==========================================
# 3. MENDEFINISIKAN KLIEN FLOWER (DIPERBAIKI)
# ==========================================
class DDoSClient(fl.client.NumPyClient):
    # PERBAIKAN: Kita memasukkan 'model' sebagai barang bawaan klien
    def __init__(self, cid, X_local, y_local, model):
        self.cid = cid
        self.X_local = X_local
        self.y_local = y_local
        self.model = model 

    def get_parameters(self, config):
        return get_model_parameters(self.model)

    def fit(self, parameters, config):
        set_model_parameters(self.model, parameters)
        for _ in range(3): 
            self.model.partial_fit(self.X_local, self.y_local)
        return get_model_parameters(self.model), len(self.X_local), {}

    def evaluate(self, parameters, config):
        set_model_parameters(self.model, parameters)
        preds = self.model.predict(self.X_local)
        acc = accuracy_score(self.y_local, preds)
        loss = float(1.0 - acc)
        return loss, len(self.X_local), {"accuracy": float(acc)}

# ==========================================
# 4. FUNGSI PEMBUAT KLIEN UNTUK SIMULATOR
# ==========================================
def client_fn(cid: str) -> fl.client.Client:
    client_key = f"client_{int(cid) + 1}"
    X_local = client_data[client_key]['X']
    y_local = client_data[client_key]['y']
    
    # PERBAIKAN: Kita fotokopi base_model dan memberikannya ke klien ini
    local_model = copy.deepcopy(base_model)
    
    return DDoSClient(cid, X_local, y_local, local_model).to_client()

# ==========================================
# 5. MENJALANKAN SERVER & SIMULASI
# ==========================================
print("--- Memulai Simulasi Flower Framework dengan Ray ---")

strategy = fl.server.strategy.FedAvg(
    fraction_fit=1.0,         
    fraction_evaluate=1.0,    
    min_fit_clients=3,        
    min_available_clients=3,
)

fl.simulation.start_simulation(
    client_fn=client_fn,
    num_clients=3,
    config=fl.server.ServerConfig(num_rounds=5),
    strategy=strategy,
    client_resources={"num_cpus": 1, "num_gpus": 0} 
)

In [67]:
pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.
